# 7.2 数据转换

## 7.2.1 删除重复数据

In [1]:
import pandas as pd
import numpy as np
data = pd.DataFrame({"k1": ["one", "two"] * 3 + ["two"],
                     "k2": [1, 1, 2, 3, 3, 4, 4]})
data

,k1,k2
0,one,1
1,two,1
2,one,2
3,two,3
4,one,3
5,two,4
6,two,4


In [2]:
# 返回bool型Series表示各行的列上的值在前面的行中出现过
data.duplicated()

0    False
1    False
2    False
3    False
4    False
5    False
6     True
dtype: bool

In [3]:
data.drop_duplicates()  # 把上面的Series里的True的行删除掉

,k1,k2
0,one,1
1,two,1
2,one,2
3,two,3
4,one,3
5,two,4


In [4]:
# 也可以指定部分列判断是否重复
data['v1'] = range(7)
data

,k1,k2,v1
0,one,1,0
1,two,1,1
2,one,2,2
3,two,3,3
4,one,3,4
5,two,4,5
6,two,4,6


In [6]:
data.drop_duplicates(subset=['k1'])

,k1,k2,v1
0,one,1,0
1,two,1,1


In [7]:
# drop_duplicates默认保留的是第一个出现的值的组合 传入keep='last'则保留最后一个
data.drop_duplicates(['k1','k2'],keep='last')

,k1,k2,v1
0,one,1,0
1,two,1,1
2,one,2,2
3,two,3,3
4,one,3,4
6,two,4,6


## 7.2.2 利用函数或映射进行数据转换

In [8]:
data = pd.DataFrame({"food": ["bacon", "pulled pork", "bacon",
                              "pastrami", "corned beef", "bacon",
                              "pastrami", "honey ham", "nova lox"],
                     "ounces": [4, 3, 12, 6, 7.5, 8, 3, 5, 6]})
data

,food,ounces
0,bacon,4.0
1,pulled pork,3.0
2,bacon,12.0
3,pastrami,6.0
4,corned beef,7.5
5,bacon,8.0
6,pastrami,3.0
7,honey ham,5.0
8,nova lox,6.0


In [9]:
meat_to_animal = {
  "bacon": "pig",
  "pulled pork": "pig",
  "pastrami": "cow",
  "corned beef": "cow",
  "honey ham": "pig",
  "nova lox": "salmon"
}

# 字典映射

data['animal'] = data['food'].map(meat_to_animal)
data

,food,ounces,animal
0,bacon,4.0,pig
1,pulled pork,3.0,pig
2,bacon,12.0,pig
3,pastrami,6.0,cow
4,corned beef,7.5,cow
5,bacon,8.0,pig
6,pastrami,3.0,cow
7,honey ham,5.0,pig
8,nova lox,6.0,salmon


In [11]:
def get_animal(x):
    return meat_to_animal[x]
data['food'].map(get_animal)

0       pig
1       pig
2       pig
3       cow
4       cow
5       pig
6       cow
7       pig
8    salmon
Name: food, dtype: str

## 7.2.3 替换值

In [12]:
data = pd.Series([1,-999,2,-99,-1000,3])
data

0       1
1    -999
2       2
3     -99
4   -1000
5       3
dtype: int64

In [13]:
data.replace(-999,np.nan)   # 不会修改原始值

0       1.0
1       NaN
2       2.0
3     -99.0
4   -1000.0
5       3.0
dtype: float64

In [15]:
data.replace([-999,-1000],np.nan)

0     1.0
1     NaN
2     2.0
3   -99.0
4     NaN
5     3.0
dtype: float64

In [16]:
data.replace([-999,-1000],[np.nan,0])

0     1.0
1     NaN
2     2.0
3   -99.0
4     0.0
5     3.0
dtype: float64

In [17]:
data.replace({-999:np.nan,-1000:1})

0     1.0
1     NaN
2     2.0
3   -99.0
4     1.0
5     3.0
dtype: float64

In [18]:
# 字符串的元素级替换
data = pd.Series(['apple', 'banana', 'apple pie', 'pineapple'])
result = data.str.replace('apple', 'orange')
print(result)

0        orange
1        banana
2    orange pie
3    pineorange
dtype: str


## 7.2.4 重命名轴索引

In [19]:
data = pd.DataFrame(np.arange(12).reshape((3, 4)),
                    index=["Ohio", "Colorado", "New York"],
                    columns=["one", "two", "three", "four"])
data

,one,two,three,four
Ohio,0,1,2,3
Colorado,4,5,6,7
New York,8,9,10,11


In [20]:
def transform(x):
    return x[:4].upper()
data.index.map(transform)

Index(['OHIO', 'COLO', 'NEW '], dtype='str')

In [21]:
data.index = data.index.map(transform)
data

,one,two,three,four
OHIO,0,1,2,3
COLO,4,5,6,7
NEW,8,9,10,11


In [22]:
# rename
df = pd.DataFrame({'A': [1, 2], 'B': [3, 4]})
print(df)
df.rename(columns={'A': '甲', 'B': '乙'}, inplace=True)
print(df)

   A  B
0  1  3
1  2  4
   甲  乙
0  1  3
1  2  4


In [23]:
data.rename(index=str.title,columns=str.upper)

,ONE,TWO,THREE,FOUR
Ohio,0,1,2,3
Colo,4,5,6,7
New,8,9,10,11


## 7.2.5 离散化和分箱

In [24]:
ages = [20,22,25,27,21,23,37,31,61,45,41,32]
bins = [18,25,35,60,100]
age_categories = pd.cut(ages,bins)
age_categories

[(18, 25], (18, 25], (18, 25], (25, 35], (18, 25], ..., (25, 35], (60, 100], (35, 60], (35, 60], (25, 35]]
Length: 12
Categories (4, interval[int64, right]): [(18, 25] < (25, 35] < (35, 60] < (60, 100]]

In [25]:
age_categories.codes

array([0, 0, 0, 1, 0, 0, 2, 1, 3, 2, 2, 1], dtype=int8)

In [27]:
age_categories.categories

IntervalIndex([(18, 25], (25, 35], (35, 60], (60, 100]], dtype='interval[int64, right]')

In [28]:
age_categories.categories[0]

Interval(18, 25, closed='right')

In [29]:
age_categories.value_counts()

(18, 25]     5
(25, 35]     3
(35, 60]     3
(60, 100]    1
Name: count, dtype: int64

In [30]:
# 左边是闭的可以通过right=False进行修改
pd.cut(ages,bins,right=False)

[[18, 25), [18, 25), [25, 35), [25, 35), [18, 25), ..., [25, 35), [60, 100), [35, 60), [35, 60), [25, 35)]
Length: 12
Categories (4, interval[int64, left]): [[18, 25) < [25, 35) < [35, 60) < [60, 100)]

In [31]:
group_names = ['Youth','YoungAdult','MiddleAged','Senior']
pd.cut(ages,bins,labels=group_names)

['Youth', 'Youth', 'Youth', 'YoungAdult', 'Youth', ..., 'YoungAdult', 'Senior', 'MiddleAged', 'MiddleAged', 'YoungAdult']
Length: 12
Categories (4, str): ['Youth' < 'YoungAdult' < 'MiddleAged' < 'Senior']

In [32]:
# 如果向pd.cut传入的不是确切的分箱边界,而是分箱的数据,则会根据数据的最小值和最大值计算得到等长的箱
data = np.random.uniform(size=20)
pd.cut(data,4,precision=2)  # precision=2表示限定小数点之后只有两位

[(0.69, 0.91], (0.47, 0.69], (0.021, 0.24], (0.24, 0.47], (0.24, 0.47], ..., (0.24, 0.47], (0.24, 0.47], (0.69, 0.91], (0.021, 0.24], (0.47, 0.69]]
Length: 20
Categories (4, interval[float64, right]): [(0.021, 0.24] < (0.24, 0.47] < (0.47, 0.69] < (0.69, 0.91]]

In [33]:
# pd.qcut使用样本分位数因此可以得到大小基本相等的分箱
data = np.random.randn(1000)
quartiles = pd.qcut(data,4,precision=2)
quartiles

[(-0.69, 0.031], (0.031, 0.63], (-3.86, -0.69], (-3.86, -0.69], (0.031, 0.63], ..., (-0.69, 0.031], (0.031, 0.63], (-3.86, -0.69], (0.63, 3.35], (0.031, 0.63]]
Length: 1000
Categories (4, interval[float64, right]): [(-3.86, -0.69] < (-0.69, 0.031] < (0.031, 0.63] < (0.63, 3.35]]

In [37]:
pd.Series(quartiles).value_counts(ascending=True)  # 将分类数据转成Series后才能在value_counts方法中传参 当热这里是为了表明qcut分类在基数上是均匀的

(-3.86, -0.69]    250
(-0.69, 0.031]    250
(0.031, 0.63]     250
(0.63, 3.35]      250
Name: count, dtype: int64

In [38]:
# 可以传递自定义的分位数(0到1之间的数值,包含端点)
pd.qcut(data,[0,0.1,0.5,0.9,1]).value_counts()

(-3.855, -1.296]    100
(-1.296, 0.031]     400
(0.031, 1.304]      400
(1.304, 3.353]      100
Name: count, dtype: int64

## 7.2.6 检测和过滤异常值

In [39]:
data = pd.DataFrame(np.random.randn(1000,4))
data.describe()

,0,1,2,3
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,-0.038499,-0.038919,-0.057156,-0.081197
std,0.988420,1.021482,0.997194,0.990067
min,-3.056343,-3.402049,-2.933779,-3.335321
25%,-0.745356,-0.737876,-0.699621,-0.718758
50%,-0.009290,-0.038174,-0.076888,-0.078117
75%,0.654186,0.637453,0.584007,0.547072
max,2.974812,3.276566,3.925580,2.946669


In [40]:
col = data[2]
col[col.abs()>3]

124    3.158576
720    3.925580
821    3.061283
979    3.116972
Name: 2, dtype: float64

In [41]:
# 要选出全部含有绝对值大于3的行 可以在bool型DataFrame中使用any方法
data[(data.abs()>3).any(axis=1)]

,0,1,2,3
118,-2.027302,3.276566,-0.195936,-0.200121
124,-0.065697,-0.893183,3.158576,-0.187340
204,-0.592914,-3.352941,-0.951107,-0.212476
246,0.567135,3.023060,0.645691,1.859170
370,0.090942,3.019528,1.341773,-1.001691
720,-0.749339,-0.430929,3.925580,-0.429632
780,1.060281,-3.402049,-0.228491,0.444430
798,-3.056343,0.171202,0.706740,-0.573682
821,-0.993391,-0.848034,3.061283,-0.770515
895,-0.004051,-3.095093,-0.940812,0.911086


In [42]:
# 讲所有值限制在[-3,3]
data[data.abs()>3] = np.sign(data)*3
data.describe()

,0,1,2,3
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,-0.038443,-0.038388,-0.058418,-0.080824
std,0.988250,1.017876,0.992859,0.988906
min,-3.000000,-3.000000,-2.933779,-3.000000
25%,-0.745356,-0.737876,-0.699621,-0.718758
50%,-0.009290,-0.038174,-0.076888,-0.078117
75%,0.654186,0.637453,0.584007,0.547072
max,2.974812,3.000000,3.000000,2.946669


## 7.2.7 置换和随机采样